# Phase 5 — Tableau Export Validation

Validate the Tableau dataset and analysis summary files before dashboard building. This notebook does not alter analytical results; it only checks file completeness, schema, missing values, coverage, data types, and readiness.


In [1]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / 'project_plan.md').exists():
    REPO_ROOT = NOTEBOOK_DIR
else:
    REPO_ROOT = NOTEBOOK_DIR.parent

DATASETS = {
    'tableau_dataset': REPO_ROOT / 'data/tableau/airline_oil_tableau_dataset.csv',
    'oil_sensitivity_summary': REPO_ROOT / 'data/processed/oil_sensitivity_summary.csv',
    'oil_shock_summary': REPO_ROOT / 'data/processed/oil_shock_summary.csv',
    'market_controlled_sensitivity_summary': REPO_ROOT / 'data/processed/oil_market_sensitivity_summary.csv',
}

REPORT_PATH = REPO_ROOT / 'data/processed/tableau_validation_report.md'
EXPECTED_AIRLINES = {'Ryanair', 'Lufthansa', 'Southwest', 'American Airlines'}
EXPECTED_FREQUENCIES = {'Weekly', 'Monthly'}

print(f'Repository root: {REPO_ROOT}')
print(f'Report path: {REPORT_PATH}')


Repository root: /home/marcusai/MyProjects/Oil vs Airline Stocks Data Analysis
Report path: /home/marcusai/MyProjects/Oil vs Airline Stocks Data Analysis/data/processed/tableau_validation_report.md


## Validation helpers


In [2]:
def normalize_strings(series):
    return set(series.dropna().astype(str).str.strip())


def check_file_exists(path):
    return path.exists() and path.is_file()


def check_required_columns(df, required_columns):
    columns = set(df.columns)
    missing = [col for col in required_columns if col not in columns]
    return missing


def check_no_missing(df):
    missing_by_column = df.isna().sum()
    missing_by_column = missing_by_column[missing_by_column > 0]
    return int(df.isna().sum().sum()), missing_by_column.to_dict()


def check_expected_values(df, column, expected_values):
    if column not in df.columns:
        return sorted(expected_values), []
    values = normalize_strings(df[column])
    missing = sorted(expected_values - values)
    extra = sorted(values - expected_values)
    return missing, extra


def check_numeric_columns(df, columns):
    results = {}
    failures = []
    for col in columns:
        if col not in df.columns:
            results[col] = 'missing column'
            failures.append(col)
            continue
        converted = pd.to_numeric(df[col], errors='coerce')
        non_numeric_count = int(converted.isna().sum() - df[col].isna().sum())
        is_numeric = non_numeric_count == 0
        results[col] = 'numeric' if is_numeric else f'{non_numeric_count} non-numeric values'
        if not is_numeric:
            failures.append(col)
    return results, failures


def check_populated(df, columns):
    results = {}
    failures = []
    for col in columns:
        if col not in df.columns:
            results[col] = 'missing column'
            failures.append(col)
            continue
        blank_count = int(df[col].astype(str).str.strip().eq('').sum())
        na_count = int(df[col].isna().sum())
        status = 'populated' if blank_count == 0 and na_count == 0 else f'{na_count} missing, {blank_count} blank'
        results[col] = status
        if blank_count > 0 or na_count > 0:
            failures.append(col)
    return results, failures


def status_from_failures(failures):
    return 'PASS' if not failures else 'FAIL'


def dataset_result(name, path):
    return {
        'name': name,
        'path': str(path.relative_to(REPO_ROOT)),
        'file_exists': check_file_exists(path),
        'row_count': None,
        'column_count': None,
        'status': 'NOT RUN',
        'checks': {},
        'warnings': [],
        'failures': [],
    }


## Load files


In [3]:
dataframes = {}
results = {}

for name, path in DATASETS.items():
    result = dataset_result(name, path)
    results[name] = result
    if not result['file_exists']:
        result['status'] = 'FAIL'
        result['failures'].append('File does not exist')
        continue
    df = pd.read_csv(path)
    dataframes[name] = df
    result['row_count'] = int(len(df))
    result['column_count'] = int(len(df.columns))
    result['checks']['dataset_not_empty'] = len(df) > 0
    if len(df) == 0:
        result['failures'].append('Dataset is empty')

row_counts = {name: result['row_count'] for name, result in results.items()}
row_counts


{'tableau_dataset': 2164,
 'oil_sensitivity_summary': 16,
 'oil_shock_summary': 8,
 'market_controlled_sensitivity_summary': 8}

## A. Tableau dataset validation


In [4]:
name = 'tableau_dataset'
result = results[name]
if result['file_exists'] and name in dataframes:
    df = dataframes[name]
    required_columns = [
        'date', 'frequency', 'asset', 'asset_type', 'region', 'business_model',
        'hedging_profile', 'return', 'sp500_return', 'excess_return', 'brent_return'
    ]
    missing_columns = check_required_columns(df, required_columns)
    result['checks']['required_columns_present'] = not missing_columns
    result['checks']['missing_required_columns'] = missing_columns
    if missing_columns:
        result['failures'].append(f'Missing required columns: {missing_columns}')

    missing_total, missing_by_column = check_no_missing(df)
    result['checks']['missing_value_total'] = missing_total
    result['checks']['missing_values_by_column'] = missing_by_column
    if missing_total > 0:
        result['failures'].append('Contains missing values')

    missing_freq, extra_freq = check_expected_values(df, 'frequency', EXPECTED_FREQUENCIES)
    result['checks']['frequency_values'] = sorted(normalize_strings(df['frequency'])) if 'frequency' in df.columns else []
    result['checks']['missing_frequencies'] = missing_freq
    result['checks']['extra_frequencies'] = extra_freq
    if missing_freq:
        result['failures'].append(f'Missing frequencies: {missing_freq}')

    missing_airlines, extra_airlines = check_expected_values(df, 'asset', EXPECTED_AIRLINES)
    result['checks']['airline_values'] = sorted(normalize_strings(df['asset'])) if 'asset' in df.columns else []
    result['checks']['missing_airlines'] = missing_airlines
    result['checks']['extra_airlines'] = extra_airlines
    if missing_airlines:
        result['failures'].append(f'Missing airlines: {missing_airlines}')

    if 'date' in df.columns:
        parsed_dates = pd.to_datetime(df['date'], errors='coerce')
        failed_dates = int(parsed_dates.isna().sum() - df['date'].isna().sum())
        result['checks']['date_parse_failures'] = failed_dates
        result['checks']['date_min'] = str(parsed_dates.min().date()) if failed_dates == 0 and not parsed_dates.empty else None
        result['checks']['date_max'] = str(parsed_dates.max().date()) if failed_dates == 0 and not parsed_dates.empty else None
        if failed_dates > 0:
            result['failures'].append(f'{failed_dates} date values failed parsing')
    else:
        result['failures'].append('Missing date column')

    numeric_results, numeric_failures = check_numeric_columns(df, ['return', 'excess_return', 'brent_return', 'sp500_return'])
    result['checks']['numeric_type_checks'] = numeric_results
    if numeric_failures:
        result['failures'].append(f'Numeric type check failed for: {numeric_failures}')

    populated_results, populated_failures = check_populated(df, ['business_model', 'region', 'hedging_profile'])
    result['checks']['categorical_population_checks'] = populated_results
    if populated_failures:
        result['failures'].append(f'Categorical fields not fully populated: {populated_failures}')

    result['status'] = status_from_failures(result['failures'])

results[name]


{'name': 'tableau_dataset',
 'path': 'data/tableau/airline_oil_tableau_dataset.csv',
 'file_exists': True,
 'row_count': 2164,
 'column_count': 11,
 'status': 'PASS',
 'checks': {'dataset_not_empty': True,
  'required_columns_present': True,
  'missing_required_columns': [],
  'missing_value_total': 0,
  'missing_values_by_column': {},
  'frequency_values': ['Monthly', 'Weekly'],
  'missing_frequencies': [],
  'extra_frequencies': [],
  'airline_values': ['American Airlines', 'Lufthansa', 'Ryanair', 'Southwest'],
  'missing_airlines': [],
  'extra_airlines': [],
  'date_parse_failures': 0,
  'date_min': '2018-01-12',
  'date_max': '2026-06-30',
  'numeric_type_checks': {'return': 'numeric',
   'excess_return': 'numeric',
   'brent_return': 'numeric',
   'sp500_return': 'numeric'},
  'categorical_population_checks': {'business_model': 'populated',
   'region': 'populated',
   'hedging_profile': 'populated'}},
 'warnings': [],
 'failures': []}

## B. Oil sensitivity summary validation


In [5]:
name = 'oil_sensitivity_summary'
result = results[name]
if result['file_exists'] and name in dataframes:
    df = dataframes[name]
    required_columns = [
        'airline', 'frequency', 'return_type', 'alpha', 'beta', 'r_squared', 'p_value',
        'interpretation', 'beta_rank', 'significant', 'beta_confidence_score'
    ]
    missing_columns = check_required_columns(df, required_columns)
    result['checks']['required_columns_present'] = not missing_columns
    result['checks']['missing_required_columns'] = missing_columns
    if missing_columns:
        result['failures'].append(f'Missing required columns: {missing_columns}')

    missing_total, missing_by_column = check_no_missing(df)
    result['checks']['missing_value_total'] = missing_total
    result['checks']['missing_values_by_column'] = missing_by_column
    if missing_total > 0:
        result['failures'].append('Contains missing values')

    missing_airlines, extra_airlines = check_expected_values(df, 'airline', EXPECTED_AIRLINES)
    result['checks']['airline_values'] = sorted(normalize_strings(df['airline'])) if 'airline' in df.columns else []
    result['checks']['missing_airlines'] = missing_airlines
    result['checks']['extra_airlines'] = extra_airlines
    if missing_airlines:
        result['failures'].append(f'Missing airlines: {missing_airlines}')

    missing_freq, extra_freq = check_expected_values(df, 'frequency', EXPECTED_FREQUENCIES)
    result['checks']['frequency_values'] = sorted(normalize_strings(df['frequency'])) if 'frequency' in df.columns else []
    result['checks']['missing_frequencies'] = missing_freq
    result['checks']['extra_frequencies'] = extra_freq
    if missing_freq:
        result['failures'].append(f'Missing frequencies: {missing_freq}')

    if 'return_type' in df.columns:
        return_types = normalize_strings(df['return_type'])
        has_return = any(value.lower() in {'return', 'returns'} for value in return_types)
        has_excess = any('excess' in value.lower() for value in return_types)
        result['checks']['return_type_values'] = sorted(return_types)
        result['checks']['contains_return_analysis'] = has_return
        result['checks']['contains_excess_return_analysis'] = has_excess
        if not has_return:
            result['failures'].append('Missing return analysis')
        if not has_excess:
            result['failures'].append('Missing excess return analysis')
    else:
        result['failures'].append('Missing return_type column')

    numeric_results, numeric_failures = check_numeric_columns(df, ['alpha', 'beta', 'r_squared', 'p_value', 'beta_rank', 'beta_confidence_score'])
    result['checks']['numeric_type_checks'] = numeric_results
    if numeric_failures:
        result['failures'].append(f'Numeric type check failed for: {numeric_failures}')

    result['status'] = status_from_failures(result['failures'])

results[name]


{'name': 'oil_sensitivity_summary',
 'path': 'data/processed/oil_sensitivity_summary.csv',
 'file_exists': True,
 'row_count': 16,
 'column_count': 11,
 'status': 'PASS',
 'checks': {'dataset_not_empty': True,
  'required_columns_present': True,
  'missing_required_columns': [],
  'missing_value_total': 0,
  'missing_values_by_column': {},
  'airline_values': ['American Airlines', 'Lufthansa', 'Ryanair', 'Southwest'],
  'missing_airlines': [],
  'extra_airlines': [],
  'frequency_values': ['Monthly', 'Weekly'],
  'missing_frequencies': [],
  'extra_frequencies': [],
  'return_type_values': ['Excess Returns', 'Returns'],
  'contains_return_analysis': True,
  'contains_excess_return_analysis': True,
  'numeric_type_checks': {'alpha': 'numeric',
   'beta': 'numeric',
   'r_squared': 'numeric',
   'p_value': 'numeric',
   'beta_rank': 'numeric',
   'beta_confidence_score': 'numeric'}},
 'warnings': [],
 'failures': []}

## C. Oil shock summary validation


In [6]:
name = 'oil_shock_summary'
result = results[name]
if result['file_exists'] and name in dataframes:
    df = dataframes[name]
    required_columns = [
        'airline', 'frequency', 'oil_shock_count',
        'average_return_during_oil_shocks', 'average_excess_return_during_oil_shocks'
    ]
    missing_columns = check_required_columns(df, required_columns)
    result['checks']['required_columns_present'] = not missing_columns
    result['checks']['missing_required_columns'] = missing_columns
    if missing_columns:
        result['failures'].append(f'Missing required columns: {missing_columns}')

    missing_total, missing_by_column = check_no_missing(df)
    result['checks']['missing_value_total'] = missing_total
    result['checks']['missing_values_by_column'] = missing_by_column
    if missing_total > 0:
        result['failures'].append('Contains missing values')

    missing_airlines, extra_airlines = check_expected_values(df, 'airline', EXPECTED_AIRLINES)
    result['checks']['airline_values'] = sorted(normalize_strings(df['airline'])) if 'airline' in df.columns else []
    result['checks']['missing_airlines'] = missing_airlines
    result['checks']['extra_airlines'] = extra_airlines
    if missing_airlines:
        result['failures'].append(f'Missing airlines: {missing_airlines}')

    missing_freq, extra_freq = check_expected_values(df, 'frequency', EXPECTED_FREQUENCIES)
    result['checks']['frequency_values'] = sorted(normalize_strings(df['frequency'])) if 'frequency' in df.columns else []
    result['checks']['missing_frequencies'] = missing_freq
    result['checks']['extra_frequencies'] = extra_freq
    if missing_freq:
        result['failures'].append(f'Missing frequencies: {missing_freq}')

    numeric_results, numeric_failures = check_numeric_columns(df, ['oil_shock_count', 'average_return_during_oil_shocks', 'average_excess_return_during_oil_shocks'])
    result['checks']['numeric_type_checks'] = numeric_results
    if numeric_failures:
        result['failures'].append(f'Numeric type check failed for: {numeric_failures}')

    if 'oil_shock_count' in df.columns:
        oil_shock_counts_numeric = pd.to_numeric(df['oil_shock_count'], errors='coerce')
        non_positive_counts = int((oil_shock_counts_numeric <= 0).sum())
        result['checks']['oil_shock_count_min'] = int(oil_shock_counts_numeric.min())
        result['checks']['oil_shock_count_max'] = int(oil_shock_counts_numeric.max())
        result['checks']['non_positive_oil_shock_counts'] = non_positive_counts
        if non_positive_counts > 0:
            result['failures'].append('Oil shock counts include non-positive values')

    result['status'] = status_from_failures(result['failures'])

results[name]


{'name': 'oil_shock_summary',
 'path': 'data/processed/oil_shock_summary.csv',
 'file_exists': True,
 'row_count': 8,
 'column_count': 5,
 'status': 'PASS',
 'checks': {'dataset_not_empty': True,
  'required_columns_present': True,
  'missing_required_columns': [],
  'missing_value_total': 0,
  'missing_values_by_column': {},
  'airline_values': ['American Airlines', 'Lufthansa', 'Ryanair', 'Southwest'],
  'missing_airlines': [],
  'extra_airlines': [],
  'frequency_values': ['Monthly', 'Weekly'],
  'missing_frequencies': [],
  'extra_frequencies': [],
  'numeric_type_checks': {'oil_shock_count': 'numeric',
   'average_return_during_oil_shocks': 'numeric',
   'average_excess_return_during_oil_shocks': 'numeric'},
  'oil_shock_count_min': 12,
  'oil_shock_count_max': 48,
  'non_positive_oil_shock_counts': 0},
 'warnings': [],
 'failures': []}

## D. Market-controlled sensitivity summary validation


In [7]:
name = 'market_controlled_sensitivity_summary'
result = results[name]
if result['file_exists'] and name in dataframes:
    df = dataframes[name]
    required_columns = [
        'airline', 'frequency', 'alpha', 'oil_beta', 'oil_p_value', 'market_beta',
        'market_p_value', 'r_squared', 'oil_significant', 'market_significant', 'interpretation'
    ]
    missing_columns = check_required_columns(df, required_columns)
    result['checks']['required_columns_present'] = not missing_columns
    result['checks']['missing_required_columns'] = missing_columns
    if missing_columns:
        result['failures'].append(f'Missing required columns: {missing_columns}')

    missing_total, missing_by_column = check_no_missing(df)
    result['checks']['missing_value_total'] = missing_total
    result['checks']['missing_values_by_column'] = missing_by_column
    if missing_total > 0:
        result['failures'].append('Contains missing values')

    missing_airlines, extra_airlines = check_expected_values(df, 'airline', EXPECTED_AIRLINES)
    result['checks']['airline_values'] = sorted(normalize_strings(df['airline'])) if 'airline' in df.columns else []
    result['checks']['missing_airlines'] = missing_airlines
    result['checks']['extra_airlines'] = extra_airlines
    if missing_airlines:
        result['failures'].append(f'Missing airlines: {missing_airlines}')

    missing_freq, extra_freq = check_expected_values(df, 'frequency', EXPECTED_FREQUENCIES)
    result['checks']['frequency_values'] = sorted(normalize_strings(df['frequency'])) if 'frequency' in df.columns else []
    result['checks']['missing_frequencies'] = missing_freq
    result['checks']['extra_frequencies'] = extra_freq
    if missing_freq:
        result['failures'].append(f'Missing frequencies: {missing_freq}')

    numeric_results, numeric_failures = check_numeric_columns(df, ['oil_beta', 'market_beta', 'oil_p_value', 'market_p_value', 'r_squared'])
    result['checks']['numeric_type_checks'] = numeric_results
    if numeric_failures:
        result['failures'].append(f'Numeric type check failed for: {numeric_failures}')

    boolean_like = {}
    for col in ['oil_significant', 'market_significant']:
        if col in df.columns:
            values = normalize_strings(df[col])
            allowed = {'True', 'False', 'true', 'false', '0', '1'}
            invalid = sorted(values - allowed)
            boolean_like[col] = {'values': sorted(values), 'invalid_values': invalid}
            if invalid:
                result['failures'].append(f'{col} contains non-boolean-like values: {invalid}')
        else:
            result['failures'].append(f'Missing {col} column')
    result['checks']['boolean_like_checks'] = boolean_like

    result['status'] = status_from_failures(result['failures'])

results[name]


{'name': 'market_controlled_sensitivity_summary',
 'path': 'data/processed/oil_market_sensitivity_summary.csv',
 'file_exists': True,
 'row_count': 8,
 'column_count': 11,
 'status': 'PASS',
 'checks': {'dataset_not_empty': True,
  'required_columns_present': True,
  'missing_required_columns': [],
  'missing_value_total': 0,
  'missing_values_by_column': {},
  'airline_values': ['American Airlines', 'Lufthansa', 'Ryanair', 'Southwest'],
  'missing_airlines': [],
  'extra_airlines': [],
  'frequency_values': ['Monthly', 'Weekly'],
  'missing_frequencies': [],
  'extra_frequencies': [],
  'numeric_type_checks': {'oil_beta': 'numeric',
   'market_beta': 'numeric',
   'oil_p_value': 'numeric',
   'market_p_value': 'numeric',
   'r_squared': 'numeric'},
  'boolean_like_checks': {'oil_significant': {'values': ['False', 'True'],
    'invalid_values': []},
   'market_significant': {'values': ['True'], 'invalid_values': []}}},
 'warnings': [],
 'failures': []}

## Build validation report


In [8]:
def format_dict(d, indent='  '):
    if not d:
        return f'{indent}- None'
    lines = []
    for key, value in d.items():
        lines.append(f'{indent}- {key}: {value}')
    return '\n'.join(lines)


def dataset_section(result):
    checks = result['checks']
    lines = [
        f"### {result['name'].replace('_', ' ').title()}",
        '',
        f"- Path: `{result['path']}`",
        f"- Validation status: **{result['status']}**",
        f"- File exists: {result['file_exists']}",
        f"- Row count: {result['row_count']}",
        f"- Column count: {result['column_count']}",
        f"- Required columns present: {checks.get('required_columns_present', 'not checked')}",
        f"- Missing required columns: {checks.get('missing_required_columns', [])}",
        f"- Missing-value total: {checks.get('missing_value_total', 'not checked')}",
    ]
    if 'missing_values_by_column' in checks:
        lines.append('- Missing values by column:')
        lines.append(format_dict(checks['missing_values_by_column']))
    if 'frequency_values' in checks:
        lines.extend([
            f"- Frequency values: {checks.get('frequency_values')}",
            f"- Missing frequencies: {checks.get('missing_frequencies')}",
            f"- Extra frequencies: {checks.get('extra_frequencies')}",
        ])
    if 'airline_values' in checks:
        lines.extend([
            f"- Airline coverage: {checks.get('airline_values')}",
            f"- Missing airlines: {checks.get('missing_airlines')}",
            f"- Extra airlines: {checks.get('extra_airlines')}",
        ])
    if 'numeric_type_checks' in checks:
        lines.append('- Numeric type checks:')
        lines.append(format_dict(checks['numeric_type_checks']))
    if 'categorical_population_checks' in checks:
        lines.append('- Business model / region / hedging profile population checks:')
        lines.append(format_dict(checks['categorical_population_checks']))
    if 'date_parse_failures' in checks:
        lines.extend([
            f"- Date parse failures: {checks.get('date_parse_failures')}",
            f"- Date range: {checks.get('date_min')} to {checks.get('date_max')}",
        ])
    if 'return_type_values' in checks:
        lines.extend([
            f"- Return type values: {checks.get('return_type_values')}",
            f"- Contains return analysis: {checks.get('contains_return_analysis')}",
            f"- Contains excess return analysis: {checks.get('contains_excess_return_analysis')}",
        ])
    if 'oil_shock_count_min' in checks:
        lines.extend([
            f"- Oil shock count range: {checks.get('oil_shock_count_min')} to {checks.get('oil_shock_count_max')}",
            f"- Non-positive oil shock counts: {checks.get('non_positive_oil_shock_counts')}",
        ])
    if 'boolean_like_checks' in checks:
        lines.append('- Significant-flag checks:')
        for col, info in checks['boolean_like_checks'].items():
            lines.append(f"  - {col}: values={info['values']}, invalid_values={info['invalid_values']}")
    if result['warnings']:
        lines.append('- Warnings:')
        for warning in result['warnings']:
            lines.append(f'  - {warning}')
    if result['failures']:
        lines.append('- Failures:')
        for failure in result['failures']:
            lines.append(f'  - {failure}')
    lines.append('')
    return '\n'.join(lines)

all_passed = all(result['status'] == 'PASS' for result in results.values())
final_status = 'READY FOR TABLEAU' if all_passed else 'NOT READY FOR TABLEAU'

warnings = [
    'The validation confirms schema, completeness, coverage, and basic type integrity; it does not re-run the underlying analytical regressions.',
    'Hedging profiles are simplified static labels and should be interpreted as exploratory classifications.',
    'Monthly analysis remains the primary interpretation layer; weekly analysis is a robustness check.'
]

report_lines = [
    '# Tableau Export Validation Report',
    '',
    f'- Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
    f'- Validation status: **{final_status}**',
    '',
    '## Row Counts',
    '',
]
for name, result in results.items():
    report_lines.append(f"- {result['path']}: {result['row_count']} rows")
report_lines.extend(['', '## Dataset Checks', ''])
for result in results.values():
    report_lines.append(dataset_section(result))
report_lines.extend([
    '## Warnings and Limitations',
    '',
])
for warning in warnings:
    report_lines.append(f'- {warning}')
report_lines.extend([
    '',
    '## Final Readiness Status',
    '',
    f'**{final_status}**',
    '',
])
if all_passed:
    report_lines.append('All required Tableau export and summary files passed validation and are ready for dashboard building.')
else:
    report_lines.append('One or more validation checks failed. Resolve the listed failures before building Tableau dashboards.')

REPORT_PATH.write_text('\n'.join(report_lines), encoding='utf-8')
print(f'Validation report written to: {REPORT_PATH}')
print(f'Final status: {final_status}')


Validation report written to: /home/marcusai/MyProjects/Oil vs Airline Stocks Data Analysis/data/processed/tableau_validation_report.md
Final status: READY FOR TABLEAU


## Final validation summary


In [9]:
summary = pd.DataFrame([
    {
        'dataset': result['name'],
        'path': result['path'],
        'status': result['status'],
        'rows': result['row_count'],
        'columns': result['column_count'],
        'failure_count': len(result['failures']),
    }
    for result in results.values()
])
print(summary.to_string(index=False))

if not all(result['status'] == 'PASS' for result in results.values()):
    raise AssertionError('One or more Tableau validation checks failed.')

print('All Phase 5 validation checks passed. Tableau datasets are ready for dashboard building.')


                              dataset                                              path status  rows  columns  failure_count
                      tableau_dataset      data/tableau/airline_oil_tableau_dataset.csv   PASS  2164       11              0
              oil_sensitivity_summary        data/processed/oil_sensitivity_summary.csv   PASS    16       11              0
                    oil_shock_summary              data/processed/oil_shock_summary.csv   PASS     8        5              0
market_controlled_sensitivity_summary data/processed/oil_market_sensitivity_summary.csv   PASS     8       11              0
All Phase 5 validation checks passed. Tableau datasets are ready for dashboard building.
